# Chat with an LLM — Insurance Submission Intake Learning Prototype

This beginner-friendly notebook demonstrates how an AI Underwriting Assistant could summarize a **synthetic** commercial insurance submission and identify documents that may need attention.

> **Prototype boundary:** This notebook is not connected to production systems, a submission repository, a broker, or a policy system. It does not make underwriting decisions. Use synthetic examples only. The LLM provider has not been selected or confirmed for this project. The optional API example below works only if an approved provider supports the documented OpenAI-compatible chat-completions format and bearer-token authentication. That compatibility has not been verified.

No API key or `.env` file is included or created by this notebook.

In [25]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Insurance AI notebook is ready!")

Python version: 3.13.16 (v3.13.16:cbc944f4bc5, Sep 30 2026, 16:01:51) [Clang 21.0.0 (clang-2100.1.1.101)]
Python executable: /Users/ganga.selvakumar/Documents/Insurance Ai Workflow/.venv/bin/python
Insurance AI notebook is ready!


In [26]:
import sys
import dotenv
import requests

print("Python:", sys.version)
print("Environment:", sys.executable)
print("Dependencies are ready!")

Python: 3.13.16 (v3.13.16:cbc944f4bc5, Sep 30 2026, 16:01:51) [Clang 21.0.0 (clang-2100.1.1.101)]
Environment: /Users/ganga.selvakumar/Documents/Insurance Ai Workflow/.venv/bin/python
Dependencies are ready!


## How to use this notebook safely

1. Open `chat_llm.ipynb` in Antigravity.
2. Select the Python kernel/environment where this notebook's dependencies will be installed. In Antigravity, use the notebook's kernel or interpreter selector; the exact label and location may vary by version.
3. Run the dependency-install cell, then run the remaining cells in order.
4. By default, **network calls are disabled**. You can run all cells safely with `ENABLE_LLM_CALL = False`; the notebook will show the prepared synthetic prompt without sending it anywhere.
5. To make an LLM request later, first choose and approve a provider, confirm its endpoint/authentication/API format and data-handling policy, configure a local `.env` as described below, then explicitly change `ENABLE_LLM_CALL` to `True`. The request still uses only the synthetic example in this notebook.

To explore one step at a time, click a cell and use the run-cell action. Use **Run All** only after reviewing the notebook: it installs dependencies and, if you explicitly enabled the call and configured the provider, sends one synthetic prompt to that provider. It never sends real customer or submission information.

## 1. Install the small set of notebook dependencies

Run this cell using the selected Python kernel. `%pip` installs packages into that kernel's environment. The notebook uses `python-dotenv` to read local configuration and `requests` for the optional OpenAI-compatible HTTP example.

In [27]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


## 2. Configure the provider only after it is selected

The project has **not** confirmed an LLM provider, model, endpoint, or credentials. Do not guess these values. If an authorized provider is selected, consult its official documentation and confirm that it supports the API format used in the request cell. The example assumes an OpenAI-compatible `POST {base_url}/chat/completions` endpoint using a bearer API key; many providers use a different format.

For local development only, create a `.env` file yourself in the project root (the same folder as this notebook). Add values supplied by the approved provider, for example:

```text
LLM_PROVIDER=REPLACE_WITH_APPROVED_PROVIDER
LLM_API_BASE_URL=REPLACE_WITH_APPROVED_OPENAI_COMPATIBLE_BASE_URL
LLM_MODEL=REPLACE_WITH_APPROVED_MODEL
LLM_API_KEY=PASTE_YOUR_LOCALLY_AUTHORIZED_KEY_HERE
```

These are placeholders, **not working settings**. Never paste a real key into the notebook, a cell output, source control, a screenshot, or Angular/browser code. Ensure `.env` is ignored by version control before adding a key. Do not share the key. This notebook does not create `.env` or verify that a provider is configured.

In [28]:
from pathlib import Path
import os

from dotenv import load_dotenv

# Read local settings without replacing environment variables already set by your shell.
PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)

# The provider/API format is intentionally left unconfirmed.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "").strip()
LLM_API_BASE_URL = os.getenv("LLM_API_BASE_URL", "").strip().rstrip("/")
LLM_MODEL = os.getenv("LLM_MODEL", "").strip()
LLM_API_KEY = os.getenv("LLM_API_KEY", "")

print("Configuration loaded from environment variables.")
print("No secret values are displayed.")

Configuration loaded from environment variables.
No secret values are displayed.


## 3. Check whether the optional provider configuration is ready

This check reports only whether values are present; it never prints the API key. A provider connection remains unverified until you explicitly enable a call and observe a successful response. The endpoint must come from the selected provider's official documentation.

In [29]:
def configuration_issues():
    issues = []
    if not LLM_PROVIDER or LLM_PROVIDER.startswith("REPLACE_"):
        issues.append("Choose an approved LLM provider and set LLM_PROVIDER.")
    if not LLM_API_BASE_URL or "REPLACE_" in LLM_API_BASE_URL:
        issues.append("Set LLM_API_BASE_URL from the approved provider's documentation.")
    if not LLM_MODEL or LLM_MODEL.startswith("REPLACE_"):
        issues.append("Set LLM_MODEL to an approved model name.")
    if not LLM_API_KEY:
        issues.append("Set LLM_API_KEY in your local .env or shell environment.")

    # Do not send an API key over plaintext HTTP, except to a local development server.
    if LLM_API_BASE_URL:
        is_local = LLM_API_BASE_URL.startswith(("http://localhost", "http://127.0.0.1"))
        if not LLM_API_BASE_URL.startswith("https://") and not is_local:
            issues.append("Use HTTPS for a remote API endpoint.")

    return issues

issues = configuration_issues()
if issues:
    print("Provider call is not ready:")
    for issue in issues:
        print(f"- {issue}")
else:
    print(f"Required settings are present for provider label: {LLM_PROVIDER}.")
    print("The API format and connection are still unverified until a request succeeds.")

Provider call is not ready:
- Choose an approved LLM provider and set LLM_PROVIDER.
- Set LLM_API_BASE_URL from the approved provider's documentation.
- Set LLM_MODEL to an approved model name.
- Set LLM_API_KEY in your local .env or shell environment.


## 4. Create a synthetic insurance submission example

All details below are fabricated for learning. They are not real people, companies, policies, brokers, or submissions. The example illustrates an administrative document check, not an assessment of insurability.

In [30]:
synthetic_submission = {
    "submission_id": "SYNTH-1001",
    "business_name": "Maple Street Bicycle Repair (synthetic)",
    "line_of_business": "Commercial Property (example only)",
    "business_description": "A small bicycle repair shop with a customer service counter.",
    "documents_received": ["Application", "Building photos"],
    "documents_to_check": ["Application", "Loss runs", "Building photos"],
    "notes": "This is a fabricated teaching example. Checklist rules are illustrative, not configured business rules.",
}

synthetic_submission

{'submission_id': 'SYNTH-1001',
 'business_name': 'Maple Street Bicycle Repair (synthetic)',
 'line_of_business': 'Commercial Property (example only)',
 'business_description': 'A small bicycle repair shop with a customer service counter.',
 'documents_received': ['Application', 'Building photos'],
 'documents_to_check': ['Application', 'Loss runs', 'Building photos'],
 'notes': 'This is a fabricated teaching example. Checklist rules are illustrative, not configured business rules.'}

## 5. Write a question and prepare a grounded prompt

Edit the example question if you like. The prompt tells the model to use only the supplied synthetic information, distinguish facts from unknowns, and avoid underwriting decisions. If the question is blank, the request function below will reject it before any network call.

In [31]:
user_question = "Summarize this submission and identify which listed documents are not shown as received."

prompt = f"""You are demonstrating a commercial insurance submission-intake assistant.
Use only the synthetic information below. Do not invent facts, claim external access,
or make an underwriting decision. Explain that document status is only an example
based on the supplied list, not a configured completeness determination.

Synthetic submission:
{synthetic_submission}

User question: {user_question}
"""

print("Synthetic question and prompt are ready. No request has been sent yet.")

Synthetic question and prompt are ready. No request has been sent yet.


## 6. Define the optional OpenAI-compatible request

This is a **provider-format placeholder**, not a claim that the project has selected a provider or that this endpoint is available. Only use it after the provider is approved and its documentation confirms compatible endpoint, authentication, request body, and response shape. If the selected provider differs, replace this adapter using that provider's official SDK/API instructions; keep the key in an environment variable and never print it.

The function uses a timeout, sends only the synthetic prompt, and reports errors without printing request headers, credentials, or raw provider response bodies.

In [32]:
import requests


def send_to_openai_compatible_chat(prompt_text: str) -> str:
    """Send one prompt to a confirmed OpenAI-compatible endpoint.

    This adapter is only an example. Confirm compatibility before use.
    """
    if not prompt_text or not prompt_text.strip():
        raise ValueError("The question is empty. Enter a question before sending.")

    missing = configuration_issues()
    if missing:
        raise RuntimeError("Provider configuration is incomplete. Review the configuration check above.")

    endpoint = f"{LLM_API_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": prompt_text}],
    }

    try:
        response = requests.post(endpoint, headers=headers, json=body, timeout=30)
    except requests.Timeout as exc:
        raise RuntimeError("The provider request timed out. No response was received.") from None
    except requests.RequestException as exc:
        raise RuntimeError("The provider request failed before a response was received.") from None

    if not response.ok:
        # Do not display the provider's raw response body; it may contain sensitive details.
        raise RuntimeError(f"The provider returned HTTP status {response.status_code}.")

    try:
        data = response.json()
        answer = data["choices"][0]["message"]["content"]
    except (ValueError, KeyError, IndexError, TypeError):
        raise RuntimeError("The response did not match the expected OpenAI-compatible format.") from None

    if not isinstance(answer, str) or not answer.strip():
        raise RuntimeError("The provider returned an empty response.")
    return answer.strip()

print("Request adapter is defined. It has not been called.")

Request adapter is defined. It has not been called.


## 7. Choose a question and explicitly control network access

The request is disabled by default so **Run All** is safe without making a network call. Keep it disabled while the provider, endpoint, credentials, and data-handling rules are unconfirmed. The prefilled prompt contains only synthetic information.

In [33]:
ENABLE_LLM_CALL = False  # Change to True only after provider approval and local configuration.

if not user_question or not user_question.strip():
    print("Please enter a non-empty question before continuing.")
elif not ENABLE_LLM_CALL:
    print("Network request skipped. Set ENABLE_LLM_CALL = True only after reviewing the provider requirements.")
    print("Prepared prompt preview (synthetic data only):")
    print(prompt)
else:
    try:
        answer = send_to_openai_compatible_chat(prompt)
    except ValueError as error:
        print(f"Input error: {error}")
    except RuntimeError as error:
        print(f"Request not completed: {error}")
    else:
        print("LLM response (based only on the synthetic example):")
        print(answer)

Network request skipped. Set ENABLE_LLM_CALL = True only after reviewing the provider requirements.
Prepared prompt preview (synthetic data only):
You are demonstrating a commercial insurance submission-intake assistant.
Use only the synthetic information below. Do not invent facts, claim external access,
or make an underwriting decision. Explain that document status is only an example
based on the supplied list, not a configured completeness determination.

Synthetic submission:
{'submission_id': 'SYNTH-1001', 'business_name': 'Maple Street Bicycle Repair (synthetic)', 'line_of_business': 'Commercial Property (example only)', 'business_description': 'A small bicycle repair shop with a customer service counter.', 'documents_received': ['Application', 'Building photos'], 'documents_to_check': ['Application', 'Loss runs', 'Building photos'], 'notes': 'This is a fabricated teaching example. Checklist rules are illustrative, not configured business rules.'}

User question: Summarize this s

## 8. What this prototype does not do

- It does not connect to production systems, a submission repository, broker communications, or policy data.
- It does not use real customer, broker, policy, or submission information.
- It does not confirm that the sample checklist is a real business checklist or that any document is truly missing.
- It does not make underwriting decisions, approve/decline risks, price coverage, bind policies, or send messages.
- A successful LLM response would only demonstrate an API call with synthetic text; it would not establish production readiness, source grounding, privacy approval, or integration availability.

Before using real data or enabling an integration, obtain the approvals and verification described in `outputs/phase-02/implementation-plan.md`.